# Assignment 4
Neural network

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.constants import c, G
import astropy.units as u

In [14]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv(x):
    sig = sigmoid(x)
    return sig * (1-sig)

def mse(y, y_true):
    """mean squared error cost"""
    sq = np.power((y_true - y), 2)
    return np.sum(sq) / (2* len(y))

def mse_deriv(y, y_true):
    return np.sum(y_true - y) / len(y)

process:
- input x to first 3 neurons
- calculate z's 1-3
- calculate y's 1-3

In [13]:
class neural_network:
    
    def __init__(self, x, y, w0, b0, eta, steps):
        """neural network that trains over `steps` steps with a learning rate of `eta`
        
        Parameters
        ----------
            x: `float`
                initial input
            y: `float`
                expected output
            w0: `float`
                initial weight values
            b0: `float`
                initial bias values
            eta: `float`
                learning rate
            steps: `int`
                epochs to train NN over
        """
        # number of neurons in each layer
        self.input_layer = 1
        self.hidden_layer = 3
        self.output_layer = 1
        
        ##-----
        # initialize weights and biases
        ##------
        # input -> hidden layer
        # weights: rows = hidden layer neurons 
        #          col = input neurons
        self.w = np.random.uniform(-1, 1, 6)
        self.b = np.zeros(4)
        self.w_in = np.zeros([self.hidden_layer, self.input_layer])
        self.b_in = np.zeros([self.hidden_layer, 1])
        # hidden -> output layer
        self.w_out = np.zeros([self.output_layer, self.hidden_layer])
        self.b_out = np.zeros([self.output_layer, 1])
        
        # track them over each timestep
        self.w = []
        self.b = []
        
        # self.w = np.zeros((6, steps))
        # self.b = np.zeros((4, steps))
        # self.w[:,0] = w0
        # self.b[:,0] = b0
        
        ### initialize results and gradients
        ### self.z = []
        self.zlist = []
        self.ylist = []
        self.grad_loss = []
        self.grad_b = []
        self.grad_w = []
        
        self.z = np.zeros((6, x.shape[0], steps))
        self.y = np.zeros((4, x.shape[0], steps))
        self.loss = np.zeros((x.shape[0], steps))
        self.grad_w = np.zeros_like(self.w)
        self.grad_b = np.zeros_like(self.b)
        
        # other values
        self.x = x
        self.ytrue = y
        self.eta = eta
        self.steps = np.arange(0, steps, 1, dtype=int)
    
    def f_z(self, x, w):
        """ z = w * x
        returns:
        --------
            z: array of shape (x rows, w columns)
            [ w1x1 w2x1 w3x1 ]
            [ w1x2 w2x2 w3x2 ]
            [ ...  ...  ...  ]
            [ w1xn w2xn w3xn ]
        """
        return np.matmul(w, x.T).T
    
    def f_y_in(self, z):
        """applies sigmoid activation function
        y = sigmoid( z(x,w) + b)
        results in a separate y for each weight, not collapsed
        
        returns:
            y: array with same shape as z (x rows, w col)
        """
        
        return sigmoid(z + self.b_in.T)
    
    def f_y_out(self, x, w):
        """applies sigmoid activation function
        y = sigmoid( sum( z(x, w) ) + b)
        z values are added up when put into sigmoid
        
        returns:
            y: array with shape (x rows, 1)
        """
        z = self.f_z(x, w)
        return sigmoid(np.sum(z, axis=1) + self.b_out)
    
    def forward(self, x, train=True):
        """propagates array x through NN layers
        
        returns:
            output: array with shape (x rows, 1)
        """
        
        # [z1 z2 z3]
        z_in = (self.w_in @ x).T
        # [y1 y2 y3]
        y_in = self.f_y_in(z_in)
        # [z4 z5 z6]
        z_out = (self.w_out.T @ x).T
        # [y4]
        y_out = self.f_y_out(y_in, self.w_out)
        
        if train: 
            # [z1 z2 z3 z4 z5 z6]
            self.zlist.append(np.hstack(z_in, z_out))
            self.ylist.append(np.hstack(y_in, y_out))
            return z_in, z_out, y_in, y_out
        return y_out
        
    def feed_forward(self, x, step, train=True):
        """ computes z and y using the given weights and biases at a particular timestep
    
        returns dictionary of z1 - z6 and y1 - y4"""
        
        # input -> hidden z's 
        z1_3 = np.matmul(self.w1, x.T)
        
        # input -> hidden z's
        
        z1 = self.w[0, step] * x
        z2 = self.w[1, step] * x
        z3 = self.w[2, step] * x
        
        # hidden layer
        y1 = sigmoid(z1 + self.b[0, step])
        y2 = sigmoid(z2 + self.b[1, step])
        y3 = sigmoid(z3 + self.b[2, step])
        
        # output layer
        # z_i = w_i * y_i
        z4 = self.w[3, step] * y1
        z5 = self.w[4, step] * y2
        z6 = self.w[5, step] * y3
        
        y4 = sigmoid(z4 + z5 + z6 + self.b[3, step])
        
        if train:
            print(z1)
            self.z[:,:, step] = np.array([z1, z2, z3, z4, z5, z6])
            self.y[:,:, step] = np.array([y1, y2, y3, y4])
            return

        return y4
    
    def gradient_descent(self, x):
        """propagates through nn layers and 
        calculates gradients of all weights and biases
        """
        
        # [z1 z2 z3], [z4 z5 z6], [y1 y2 y3], [y4]
        z_in, z_out, y_in, y_out = self.feed_forward(x)
        
        # output loss
        loss = mse(y_out, self.ytrue)
        dldy_out = mse_deriv(y_out, self.ytrue)
        # bias b4
        # same shape as y_out now...
        dldb_out = dldy_out * sigmoid_deriv(y_out)
        # hidden -> output weights
        dldw_out = dldb_out * y_out
        # hidden biases
        dldb_in = dldw_out * self.w_out * (1 - y_in) * y_in
        # input -> hidden weights
        dldw_in = dldb_in * x
        
        self.grad_loss.append(loss)
        self.grad_w.append(np.hstack(dldw_in, dldw_out))
        self.grad_b.append(np.hstack(dldb_in, dldb_out))
        
        return y_out
    
    def grad_descent(self, step):
        
        # output loss
        dldy4 = self.y[3,:, step] - self.ytrue
        # bias b4
        dldb4 = dldy4 * self.y[3,:, step] * (1-self.y[3,:, step])

        # hidden -> output weights
        dldw4 = dldb4 * self.y[0,:, step]
        dldw5 = dldb4 * self.y[1,:, step]
        dldw6 = dldb4 * self.y[2,:, step]

        # hidden biases
        dldb1 = dldw4 * (1-self.y[0,:, step]) * self.w[3,step]
        dldb2 = dldw5 * (1-self.y[1,:, step]) * self.w[4,step]
        dldb3 = dldw6 * (1-self.y[2,:, step]) * self.w[5,step]

        # hidden weights
        dldw1 = dldb1 * self.x
        dldw2 = dldb2 * self.x
        dldw3 = dldb3 * self.x

        self.loss[:,step] = dldy4
        self.grad_w[:,step] = np.array([dldw1, dldw2, dldw3, dldw4, dldw5, dldw6])
        self.grad_b[:,step] = np.array([dldb1, dldb2, dldb3, dldb4])
    
    def learning(self, step):
        """ applies gradient descent and learning to biases and weights at a given timestep
        w(t+1) = w(t) - grad(L) * eta
        """
        
        #weights
        self.w.append(np.hstack(self.w_in, self.w_out))
        self.b.append(np.hstack(self.b_in, self.b_out))

        new_w = self.w[-1] - self.grad_w[-1] * self.eta
        new_b = self.b[-1] - self.grad_b[-1] * self.eta
        
        self.w_in = new_w[:, :3]
        self.w_out = new_w[:, 3:]
        self.b_in = new_b[:, :3]
        self.b_out = new_b[:,-1]
        # weights
        for i, grad_weight in enumerate(self.grad_w):
            self.w[i, step+1] = self.w[i, step] - grad_weight[step] * self.eta

        # biases
        for i, grad_bias in enumerate(self.grad_b):
            self.b[i, step+1] = self.b[i, step] - grad_bias[step] * self.eta
            
    def train_network(self):
        """ trains the network over `steps` steps"""
        
        for i in self.steps:
            self.feed_forward(self.x,i)
            self.grad_descent(i)
            if i < self.steps[-1]:
                self.learning(i)
                
        print(f"Predicted y: {self.y[3, -1]:.4f}")
        print(f"Actual y:    {self.ytrue:.4f}")
        
    def plot_network(self):
        fig, ax = plt.subplots(1,2, figsize=(8,4))
        # output
        ax[0].plot(self.steps, self.y[3], label='output y')
        ax[0].hlines(self.ytrue, xmin=self.steps[0], xmax=self.steps[-1], color='r', ls='--', label='expected y')
        ax[0].set(xlabel='steps', ylabel='y', title=f'output for x={self.x}')
        ax[0].legend()
        # loss
        ax[1].plot(self.steps, self.loss)
        ax[1].set(xlabel='steps', ylabel='loss', title='loss function')
        
        plt.tight_layout()
        plt.show()
        
    def predict(self, x):
        return self.feed_forward(x, self.steps[-1], train=False)
        


In [ ]:
def stellar_lifetime(m, alpha=3.5, p_ms=0.1):
    """returns the total lifetime of a star of mass m
    
    parameters
    ----------
        m: `float`
            mass of star (Msun)
        alpha: `float`
            power law index of stellar mass-lifetime (default = 3.5)
        p_ms: `float`
            percentage of lifetime spent post-main sequence (default = 0.1)
    
    returns
    -------
        t: `float`
            stellar lifetime from zero-age-main sequence to stellar remnant (Gyr)
    """
    t_sun = 1 * u.Gyr
    return (1 + p_ms) * t_sun * m**(-alpha)
    
    

In [10]:
rng = np.random.default_rng()
x = rng.uniform(0.1, 10, 10) # Msun
y = (2 * G * x*u.Msun / c**2).to(u.km).value # meters
# print(y)
# x = 1
# y = 2
w0 = rng.uniform(-5, 5, 6)
b0 = rng.uniform(-5, 5, 4)
eta = 0.1
nsteps = 100

print(f"weights: {w0}")
print(f"biases: {b0}")
network = neural_network(x, y, w0, b0, eta, nsteps)
output = network.forward(x)
# network.train_network()
network.plot_network()

# xguess = np.arange(5)
# yguess = network.predict(xguess)
# fig, ax = plt.subplots()
# ax.plot(xguess, yguess)
# ax.set(xlabel="x", ylabel="y")
# plt.show()


weights: [-0.06424858  0.93335681 -4.58916631  1.24741534  2.33479219 -4.76597586]
biases: [ 1.12955579  1.89969585 -0.85082282 -0.05109832]


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 10 is different from 1)

In [37]:
# input -> hidden
print(r"| step | $w_1$ | $\nabla L(w_{1})$ | $z_1$ | $w_2$ | $\nabla L(w_{2})$ | $z_2$ | $w_3$ | $\nabla L(w_{3})$ | $z_3$ | $b_1$ | $\nabla L(b_{1})$ | $b_2$ | $\nabla L(b_{2})$ | $b_3$ | $\nabla L(b_{3})$ | $y_1$ | $y_2$ | $y_3$ |")
print(r"| ---- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----------------- | ----- | ----------------- | ----- | ---- | ---- |")
print(f"|1| {w[0]:.4f} | {grad['dldw1']:.4f} | {z1:.4f} | {w[1]:.4f} | {grad['dldw2']:.4f} | {z2:.4f} | {w[2]:.4f} | {grad['dldw3']:.4f} | {z3:.4f} | \
{b[0]:.4f} | {grad['dldb1']:.4f} | {b[1]:.4f} | {grad['dldb2']:.4f} | {b[3]:.4f} | {grad['dldb3']:.4f} | {y1:.4f} | {y2:.4f} | {y3:.4f} |")

| step | $w_1$ | $\nabla L(w_{1})$ | $z_1$ | $w_2$ | $\nabla L(w_{2})$ | $z_2$ | $w_3$ | $\nabla L(w_{3})$ | $z_3$ | $b_1$ | $\nabla L(b_{1})$ | $b_2$ | $\nabla L(b_{2})$ | $b_3$ | $\nabla L(b_{3})$ | $y_1$ | $y_2$ | $y_3$ |
| ---- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----------------- | ----- | ----------------- | ----- | ---- | ---- |
|1| 0.5000 | 0.0004 | 0.5000 | 0.5000 | 0.0004 | 0.5000 | 0.5000 | 0.0004 | 0.5000 | 0.5000 | 0.0004 | 0.5000 | 0.0004 | 0.5000 | 0.0004 | 0.7311 | 0.7311 | 0.7311 |


In [38]:
# hidden -> output
print(r"| step | $w_4$ | $\nabla L(w_{4})$ | $z_4$ | $w_5$ | $\nabla L(w_{5})$ | $z_5$ | $w_6$ | $\nabla L(w_{6})$ | $z_6$ | $b_4$ | $\nabla L(b_{4})$ | $y_4$ | $L(y_4)$ |")
print(r"| ---- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | -------- |")
print(f"|1| {w[3]:.4f} | {grad['dldw4']:.4f} | {z4:.4f} | {w[4]:.4f} | {grad['dldw5']:.4f} | {z5:.4f} | {w[5]:.4f} | {grad['dldw6']:.4f} |\
{z6:.4f} | {b[3]:.4f} | {grad['dldb4']:.4f}| {y4:.4f} | {grad['dldy4']:.4f} |")

| step | $w_4$ | $\nabla L(w_{4})$ | $z_4$ | $w_5$ | $\nabla L(w_{5})$ | $z_5$ | $w_6$ | $\nabla L(w_{6})$ | $z_6$ | $b_4$ | $\nabla L(b_{4})$ | $y_4$ | $L(y_4)$ |
| ---- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | -------- |
|1| 0.5000 | 0.0029 | 0.3655 | 0.5000 | 0.0029 | 0.3655 | 0.5000 | 0.0029 |0.3655 | 0.5000 | -0.0236| 0.8315 | -0.1685 |


imput to hidden layer

| step | $w_1$ | $\nabla L(w_{1})$ | $z_1$ | $w_2$ | $\nabla L(w_{2})$ | $z_2$ | $w_3$ | $\nabla L(w_{3})$ | $z_3$ | $b_1$ | $\nabla L(b_{1})$ | $b_2$ | $\nabla L(b_{2})$ | $b_3$ | $\nabla L(b_{3})$ | $y_1$ | $y_2$ | $y_3$ |
| ---- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----------------- | ----- | ----------------- | ----- | ---- | ---- |
|1| 0.5000 | 0.0188 | 0.5000 | 0.5000 | 0.0188 | 0.5000 | 0.5000 | 0.0188 | 0.5000 | 0.5000 | 0.0188 | 0.5000 | 0.0188 | 0.5000 | 0.0188 | 0.7311 | 0.7311 | 0.7311 |

hidden layer to output layer

| step | $w_4$ | $\nabla L(w_{4})$ | $z_4$ | $w_5$ | $\nabla L(w_{5})$ | $z_5$ | $w_6$ | $\nabla L(w_{6})$ | $z_6$ | $b_4$ | $\nabla L(b_{4})$ | $y_4$ | $L(y_4)$ |
| ---- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | ----- | ----------------- | ----- | -------- |
|1| 0.5000 | 0.1398 | 0.3655 | 0.5000 | 0.1398 | 0.3655 | 0.5000 | 0.1398 |0.3655 | 0.5000 | -0.1637| 0.8315 | -1.1685 |